In [ ]:
# temp

## 04. Binary + Masked Shade Ablation Evaluation

This notebook evaluates the binary+shade ablation model:
- `bin_head`: main binary metrics (F1/AUC/tuned-threshold)
- `shade_head`: auxiliary diagnostics with conditional masking by `walking_paths_p`


In [1]:
# Setup: imports, paths, and dataframes
import pandas as pd
import numpy as np
import tensorflow as tf
from pathlib import Path

# Evaluate on the saved split manifests (created in 02_data_preprocessing.ipynb)
splits_dir = Path('../data/processed/splits')
train_csv = splits_dir / 'train.csv'
val_csv   = splits_dir / 'val.csv'
test_csv  = splits_dir / 'test.csv'

for p in [train_csv, val_csv, test_csv]:
    assert p.exists(), f"Missing split manifest: {p} (run 02 first)"

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)
test_df  = pd.read_csv(test_csv)

print('Loaded splits:', {"train": len(train_df), "val": len(val_df), "test": len(test_df)})

# Binary labels are stored as probabilities in *_p columns
binary_cols = [c for c in train_df.columns if c.endswith('_p')]
assert binary_cols, 'No *_p binary prob cols found in split manifests'

# Required columns for this ablation
for df_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    for c in ['shade_class', 'image_path']:
        assert c in df.columns, f"Missing {c} in {df_name}.csv"

print('Binary prob cols:', binary_cols)
print('Aux class cols  :', ['shade_class'])


/Users/starsrain/2025_codeProject/GreenSpace_CNN/.venv/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Loaded splits: {'train': 2347, 'val': 782, 'test': 782}
Binary prob cols: ['sports_field_p', 'multipurpose_open_area_p', 'children_s_playground_p', 'water_feature_p', 'gardens_p', 'walking_paths_p', 'built_structures_p', 'parking_lots_p']
Aux class cols  : ['shade_class']


In [2]:
# Build datasets (no augmentation)
IMG_SIZE = (512, 512)
BATCH_SIZE = 8
NUM_SHADE = 2

def decode_image(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def get_shade_sample_weight(df):
    if 'walking_paths_p' in df.columns:
        return df['walking_paths_p'].fillna(0.0).astype(np.float32).values
    if 'walking_paths' in df.columns:
        return df['walking_paths'].fillna(0).astype(np.float32).values
    raise ValueError('Need walking_paths_p (preferred) or walking_paths for shade mask')

def make_ds(df):
    paths = df['image_path'].astype(str).tolist()

    ds_paths = tf.data.Dataset.from_tensor_slices(paths)
    ds_imgs = ds_paths.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)

    # labels (match ablation training)
    y_bin = df[binary_cols].fillna(0.0).astype(np.float32).values
    y_shade = df['shade_class'].fillna(0).astype(np.int32).values
    y_shade = np.clip(y_shade, 0, NUM_SHADE - 1)

    shade_sw = get_shade_sample_weight(df)
    ones = np.ones((len(paths),), dtype=np.float32)

    ds_labels = tf.data.Dataset.from_tensor_slices({
        'bin_head': y_bin,
        'shade_head': y_shade,
    })
    ds_sw = tf.data.Dataset.from_tensor_slices({
        'bin_head': ones,
        'shade_head': shade_sw,
    })

    # Include sample weights so monitoring loss reflects masked shade objective
    return tf.data.Dataset.zip((ds_imgs, ds_labels, ds_sw)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

print('Shade mask mean (train/val/test):',
      float(np.mean(get_shade_sample_weight(train_df))),
      float(np.mean(get_shade_sample_weight(val_df))),
      float(np.mean(get_shade_sample_weight(test_df))))
print('Datasets ready:', {"train": len(train_df), "val": len(val_df), "test": len(test_df)})


Shade mask mean (train/val/test): 0.7281990647315979 0.7291133999824524 0.7329496741294861
Datasets ready: {'train': 2347, 'val': 782, 'test': 782}


In [3]:
# Load a trained binary ablation model
# Preferred layout: models/runs/binary_ablation_<RUN_TAG>/final_binary_ablation_<RUN_TAG>.keras

runs_root = Path('../models/runs')

# ---- Toggle: manually load a specific model file ----
USE_MANUAL_MODEL = False
MANUAL_MODEL_PATH = None  # e.g., '../models/runs/binary_ablation_20260207_120000/final_binary_ablation_20260207_120000.keras'

if USE_MANUAL_MODEL:
    assert MANUAL_MODEL_PATH, 'Set MANUAL_MODEL_PATH when USE_MANUAL_MODEL=True'
    model_path = Path(MANUAL_MODEL_PATH)
    assert model_path.exists(), f"Manual model not found: {model_path}"
    model = tf.keras.models.load_model(str(model_path))
    print('Loaded MANUAL model from', model_path)
else:
    # default: latest binary_ablation_* run dir
    RUN_DIR = globals().get('RUN_DIR', None)
    if RUN_DIR is None:
        if runs_root.exists():
            run_dirs = [p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith('binary_ablation_')]
            run_dirs = sorted(run_dirs, key=lambda p: p.stat().st_mtime, reverse=True)
            RUN_DIR = run_dirs[0] if run_dirs else (runs_root / 'binary_ablation_REPLACE_WITH_RUN_TAG')
        else:
            RUN_DIR = runs_root / 'binary_ablation_REPLACE_WITH_RUN_TAG'

    RUN_DIR = Path(RUN_DIR)
    print('Using RUN_DIR =', RUN_DIR)

    candidates = []
    if RUN_DIR.exists():
        candidates += sorted(RUN_DIR.glob('final_binary_ablation_*.keras'))
        if not candidates:
            candidates += sorted(RUN_DIR.glob('best_binary_ablation_*.keras'))

    assert candidates, (
        f"No ablation model .keras found in RUN_DIR={RUN_DIR}. "
        f"(Expected final_binary_ablation_*.keras or best_binary_ablation_*.keras)"
    )

    model_path = candidates[-1]
    model = tf.keras.models.load_model(str(model_path))
    print('Loaded model from', model_path)


Using RUN_DIR = ../models/runs/binary_ablation_20260223_201201
Loaded model from ../models/runs/binary_ablation_20260223_201201/final_binary_ablation_20260223_201201.keras


## Label Loss Monitoring

In [4]:
# Monitoring: per-head losses + metrics (train / val / test)
# For ablation model, only bin_head + shade_head are present.

losses = {
    'bin_head': 'binary_crossentropy',
    'shade_head': 'sparse_categorical_crossentropy',
}
metrics = {
    'bin_head': ['binary_accuracy'],
    'shade_head': ['sparse_categorical_accuracy'],
}
model.compile(optimizer=tf.keras.optimizers.Adam(), loss=losses, metrics=metrics)

# Infer run tag from folder/filename
run_tag = None
try:
    p = Path(model_path)
    if 'runs' in p.parts:
        runs_idx = p.parts.index('runs')
        if runs_idx + 1 < len(p.parts):
            run_tag = p.parts[runs_idx + 1]
    if run_tag is None:
        name = p.name
        if name.startswith('final_binary_ablation_') and name.endswith('.keras'):
            run_tag = name[len('final_binary_ablation_'):-len('.keras')]
except Exception:
    pass
print('Model run tag:', run_tag)

def eval_split(split_name, ds):
    d = model.evaluate(ds, verbose=0, return_dict=True)
    d['split'] = split_name
    return d

rows = [
    eval_split('train', train_ds),
    eval_split('val',   val_ds),
    eval_split('test',  test_ds),
]
mon = pd.DataFrame(rows).set_index('split')

keep = [
    'loss',
    'bin_head_loss', 'shade_head_loss',
    'bin_head_binary_accuracy',
    'shade_head_sparse_categorical_accuracy',
]
keep = [k for k in keep if k in mon.columns]

display(mon[keep].round(4))
print('Note: shade_head loss uses conditional sample weighting via walking_paths_p in datasets.')


Model run tag: binary_ablation_20260223_201201


,loss,bin_head_loss,shade_head_loss,bin_head_binary_accuracy,shade_head_sparse_categorical_accuracy
split,,,,,
train,0.2971,0.2073,0.0898,0.9016,0.9105
val,1.5939,0.3253,1.2679,0.8558,0.6662
test,1.5908,0.3358,1.2529,0.8592,0.6867


Note: shade_head loss uses conditional sample weighting via walking_paths_p in datasets.


### Save Loss Monitoring Artifact

In [5]:
# Save monitoring table
from datetime import datetime

out_dir = (Path('../monitoring_output')).resolve()
out_dir.mkdir(parents=True, exist_ok=True)

tag = run_tag or datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = out_dir / f"loss_monitor_binary_ablation_{tag}.csv"

mon[keep].to_csv(out_path)
print('Saved monitoring table to', out_path)


Saved monitoring table to /Users/starsrain/2025_codeProject/GreenSpace_CNN/monitoring_output/loss_monitor_binary_ablation_binary_ablation_20260223_201201.csv


## Threshold Tuning

In [6]:
# Threshold tuning (binary heads): tune on VAL to maximize F1, then apply to TEST
# This does NOT require retraining.

from sklearn.metrics import precision_recall_curve
from datetime import datetime

# 1) Predict on validation (ablation outputs: bin, shade)
preds_val = model.predict(val_ds, verbose=0)
if isinstance(preds_val, dict):
    pred_bin_val = preds_val['bin_head']
    pred_shade_val = preds_val['shade_head']
else:
    pred_bin_val, pred_shade_val = preds_val

# Align y_true (val) to the same binary label order used for probabilities
bin_names = [c[:-2] for c in binary_cols]
hard_bin_names_val = [c for c in bin_names if c in val_df.columns]

if hard_bin_names_val:
    y_bin_val_true = val_df[hard_bin_names_val].fillna(0).astype(int).values
    pred_bin_val_aligned = np.stack([pred_bin_val[:, bin_names.index(n)] for n in hard_bin_names_val], axis=1)
    label_names = hard_bin_names_val
else:
    # Fallback: if hard 0/1 columns are not present, create pseudo-hard labels from *_p
    y_bin_val_true = (val_df[binary_cols].fillna(0.0).astype(np.float32).values >= 0.5).astype(int)
    pred_bin_val_aligned = pred_bin_val
    label_names = bin_names

print('Tuning thresholds on VAL for labels:', label_names)


def tune_thresholds_f1(y_true_mat, y_prob_mat, label_names, min_pos=1):
    rows = []
    for i, name in enumerate(label_names):
        y_true = np.asarray(y_true_mat[:, i]).astype(int)
        y_prob = np.asarray(y_prob_mat[:, i]).astype(float)

        n_pos = int(y_true.sum())
        n = int(len(y_true))
        pos_rate = float(y_true.mean()) if n else float('nan')

        if np.unique(y_true).size < 2 or n_pos < min_pos:
            rows.append({
                'label': name,
                'best_threshold': np.nan,
                'best_f1': np.nan,
                'best_precision': np.nan,
                'best_recall': np.nan,
                'pos_rate': pos_rate,
                'n_pos': n_pos,
                'n': n,
                'note': 'single-class' if np.unique(y_true).size < 2 else 'too-few-positives',
            })
            continue

        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        if thresholds.size == 0:
            rows.append({
                'label': name,
                'best_threshold': np.nan,
                'best_f1': np.nan,
                'best_precision': np.nan,
                'best_recall': np.nan,
                'pos_rate': pos_rate,
                'n_pos': n_pos,
                'n': n,
                'note': 'no-thresholds',
            })
            continue

        p = precision[:-1]
        r = recall[:-1]
        t = thresholds
        f1 = (2 * p * r) / (p + r + 1e-12)

        best_f1 = float(np.max(f1))
        best_idxs = np.flatnonzero(f1 == best_f1)
        best_idx = int(best_idxs[0])

        rows.append({
            'label': name,
            'best_threshold': float(t[best_idx]),
            'best_f1': best_f1,
            'best_precision': float(p[best_idx]),
            'best_recall': float(r[best_idx]),
            'pos_rate': pos_rate,
            'n_pos': n_pos,
            'n': n,
            'note': '',
        })

    return pd.DataFrame(rows)


thresh_df = tune_thresholds_f1(y_bin_val_true, pred_bin_val_aligned, label_names)

best_thresholds = {
    row['label']: float(row['best_threshold'])
    for _, row in thresh_df.iterrows()
    if np.isfinite(row['best_threshold'])
}

print('\nTop thresholds by best_f1 (VAL):')
display(thresh_df.sort_values('best_f1', ascending=False).reset_index(drop=True))

_defined = thresh_df['best_f1'].notna()
if _defined.any():
    print(
        f"overall (VAL) over definable labels: "
        f"F1={float(thresh_df.loc[_defined, 'best_f1'].mean()):.3f} | "
        f"P={float(thresh_df.loc[_defined, 'best_precision'].mean()):.3f} | "
        f"R={float(thresh_df.loc[_defined, 'best_recall'].mean()):.3f}"
    )

# Save thresholds
out_dir = Path('../monitoring_output').resolve()
out_dir.mkdir(parents=True, exist_ok=True)
_tag = globals().get('run_tag', None) or datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = out_dir / f"thresholds_binary_ablation_{_tag}.csv"
thresh_df.to_csv(out_path, index=False)
print('Saved thresholds to', out_path)

Tuning thresholds on VAL for labels: ['sports_field', 'multipurpose_open_area', 'children_s_playground', 'water_feature', 'gardens', 'walking_paths', 'built_structures', 'parking_lots']

Top thresholds by best_f1 (VAL):


,label,best_threshold,best_f1,best_precision,best_recall,pos_rate,n_pos,n,note
0,multipurpose_open_area,0.087746,0.943542,0.911809,0.977564,0.797954,624,782,
1,walking_paths,0.154264,0.921423,0.885533,0.960345,0.741688,580,782,
2,built_structures,0.272597,0.828080,0.796143,0.862687,0.428389,335,782,
3,sports_field,0.603065,0.800000,0.857143,0.750000,0.286445,224,782,
4,parking_lots,0.248043,0.788571,0.766667,0.811765,0.326087,255,782,
5,water_feature,0.310659,0.568345,0.626984,0.519737,0.194373,152,782,
6,children_s_playground,0.146437,0.492647,0.370166,0.736264,0.116368,91,782,
7,gardens,0.090471,0.250000,0.180556,0.406250,0.040921,32,782,


overall (VAL) over definable labels: F1=0.699 | P=0.674 | R=0.753
Saved thresholds to /Users/starsrain/2025_codeProject/GreenSpace_CNN/monitoring_output/thresholds_binary_ablation_binary_ablation_20260223_201201.csv


In [7]:
# Thresholds should be tuned on VAL in the previous cell (best_thresholds).

from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
)

bin_names = [c[:-2] for c in binary_cols]
thresholds = best_thresholds if ('best_thresholds' in globals() and isinstance(best_thresholds, dict)) else {}


def _extract_preds(ds):
    preds = model.predict(ds, verbose=0)
    if isinstance(preds, dict):
        return preds['bin_head'], preds['shade_head']
    return preds


def _eval_overall(split_name, df, ds, thresholds_dict):
    pred_bin_local, pred_shade_local = _extract_preds(ds)
    hard_names_local = [c for c in bin_names if c in df.columns]

    if hard_names_local:
        y_bin_true_local = df[hard_names_local].fillna(0).astype(int).values
        pred_bin_aligned_local = np.stack([pred_bin_local[:, bin_names.index(n)] for n in hard_names_local], axis=1)
    else:
        y_bin_true_local = (df[binary_cols].fillna(0.0).astype(np.float32).values >= 0.5).astype(int)
        pred_bin_aligned_local = pred_bin_local
        hard_names_local = bin_names

    f1_05_list = []
    roc_list = []
    ap_list = []
    f1_tuned_list = []
    label_rows = []

    for i, name in enumerate(hard_names_local):
        y_true = y_bin_true_local[:, i]
        y_prob = pred_bin_aligned_local[:, i]

        y_hat_05 = (y_prob >= 0.5).astype(int)
        p05, r05, f105, _ = precision_recall_fscore_support(y_true, y_hat_05, average='binary', zero_division=0)
        f1_05_list.append(float(f105))

        roc = np.nan
        ap = np.nan
        if np.unique(y_true).size >= 2:
            roc = float(roc_auc_score(y_true, y_prob))
            ap = float(average_precision_score(y_true, y_prob))
            roc_list.append(roc)
            ap_list.append(ap)

        thr = thresholds_dict.get(name, np.nan)
        pt = rt = f1t = np.nan
        if np.isfinite(thr):
            y_hat_t = (y_prob >= float(thr)).astype(int)
            pt, rt, f1t, _ = precision_recall_fscore_support(y_true, y_hat_t, average='binary', zero_division=0)
            pt, rt, f1t = float(pt), float(rt), float(f1t)
            f1_tuned_list.append(f1t)

        label_rows.append({
            'split': split_name,
            'label': name,
            'support_pos': int(np.sum(y_true == 1)),
            'support_neg': int(np.sum(y_true == 0)),
            'P@0.5': float(p05),
            'R@0.5': float(r05),
            'F1@0.5': float(f105),
            'ROC_AUC': roc,
            'PR_AUC': ap,
            'thr_val': (float(thr) if np.isfinite(thr) else np.nan),
            'P@thr': pt,
            'R@thr': rt,
            'F1@thr': f1t,
        })

    y_shade_true_local = df['shade_class'].fillna(0).astype(int).values
    shade_pred = pred_shade_local.argmax(axis=1)
    shade_acc_local = float(accuracy_score(y_shade_true_local, shade_pred))

    shade_acc_conditional_local = np.nan
    shade_acc_paths_present_local = np.nan
    if 'walking_paths_p' in df.columns:
        w = df['walking_paths_p'].fillna(0.0).to_numpy(dtype=np.float32)
    elif 'walking_paths' in df.columns:
        w = df['walking_paths'].fillna(0).to_numpy(dtype=np.float32)
    else:
        w = None

    if w is not None:
        shade_correct = (shade_pred == y_shade_true_local).astype(np.float32)
        denom = float(np.sum(w))
        if denom > 1e-8:
            shade_acc_conditional_local = float(np.sum(w * shade_correct) / denom)
        mask = w >= 0.5
        if int(np.sum(mask)) > 0:
            shade_acc_paths_present_local = float(accuracy_score(y_shade_true_local[mask], shade_pred[mask]))

    overall_row = {
        'split': split_name,
        'n_samples': int(len(df)),
        'n_labels': int(len(hard_names_local)),
        'overall_F1@0.5': (float(np.mean(f1_05_list)) if f1_05_list else np.nan),
        'overall_ROC_AUC': (float(np.mean(roc_list)) if roc_list else np.nan),
        'overall_PR_AUC': (float(np.mean(ap_list)) if ap_list else np.nan),
        'overall_F1@tuned': (float(np.mean(f1_tuned_list)) if f1_tuned_list else np.nan),
        'shade_acc_overall': shade_acc_local,
        'shade_acc_conditional': shade_acc_conditional_local,
        'shade_acc_paths_present': shade_acc_paths_present_local,
    }

    return overall_row, pd.DataFrame(label_rows)


split_spec = [
    ('train', train_df, train_ds),
    ('val', val_df, val_ds),
    ('test', test_df, test_ds),
]

overall_rows = []
label_parts = []
for split_name, split_df, split_ds in split_spec:
    o_row, l_df = _eval_overall(split_name, split_df, split_ds, thresholds)
    overall_rows.append(o_row)
    label_parts.append(l_df)

overall_split_df = pd.DataFrame(overall_rows)
overall_split_df['split'] = pd.Categorical(overall_split_df['split'], categories=['train', 'val', 'test'], ordered=True)
overall_split_df = overall_split_df.sort_values('split').reset_index(drop=True)

per_label_split_df = pd.concat(label_parts, ignore_index=True)
per_label_split_df['split'] = pd.Categorical(per_label_split_df['split'], categories=['train', 'val', 'test'], ordered=True)
per_label_split_df = per_label_split_df.sort_values(['split', 'F1@thr', 'F1@0.5'], ascending=[True, False, False]).reset_index(drop=True)

print('Computed split-wise evaluation tables.')
print(f"Threshold coverage (val-tuned labels): {len(thresholds)} / {len(bin_names)}")
print('\n--- Overall metrics by split ---')
display(overall_split_df.round(4))
print('\n--- Per-label metrics by split ---')
display(per_label_split_df.round(4))

# Continue with the original TEST-specific detailed printout below.
preds_test = model.predict(test_ds, verbose=0)
if isinstance(preds_test, dict):
    pred_bin = preds_test['bin_head']
    pred_shade = preds_test['shade_head']
else:
    pred_bin, pred_shade = preds_test

# Split-wise table is shown above.
# Ground truth for binaries
bin_names = [c[:-2] for c in binary_cols]
hard_bin_names = [c for c in bin_names if c in test_df.columns]

if hard_bin_names:
    y_bin_true = test_df[hard_bin_names].fillna(0).astype(int).values
    pred_bin_aligned = np.stack([pred_bin[:, bin_names.index(n)] for n in hard_bin_names], axis=1)
else:
    y_bin_true = (test_df[binary_cols].fillna(0.0).astype(np.float32).values >= 0.5).astype(int)
    pred_bin_aligned = pred_bin
    hard_bin_names = bin_names

y_shade_true = test_df['shade_class'].fillna(0).astype(int).values

from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
)

print('--- Binary (threshold=0.5) ---')
for i, name in enumerate(hard_bin_names):
    y_prob = pred_bin_aligned[:, i]
    y_hat = (y_prob >= 0.5).astype(int)
    y_true = y_bin_true[:, i]
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_hat, average='binary', zero_division=0)
    print(f"{name:24s} P={p:.2f} R={r:.2f} F1={f1:.2f}")

f1_list_05 = []
for i, name in enumerate(hard_bin_names):
    y_prob = pred_bin_aligned[:, i]
    y_hat = (y_prob >= 0.5).astype(int)
    y_true = y_bin_true[:, i]
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_hat, average='binary', zero_division=0)
    f1_list_05.append(float(f1))
print(f"overall F1 (threshold=0.5) = {float(np.mean(f1_list_05)):.3f}")

print('--- Binary (AUC) ---')
roc_list = []
ap_list = []
for i, name in enumerate(hard_bin_names):
    y_true = y_bin_true[:, i]
    y_prob = pred_bin_aligned[:, i]

    if np.unique(y_true).size < 2:
        print(f"{name:24s} ROC_AUC=NA PR_AUC=NA (single-class)")
        continue

    roc = float(roc_auc_score(y_true, y_prob))
    ap = float(average_precision_score(y_true, y_prob))
    roc_list.append(roc)
    ap_list.append(ap)
    print(f"{name:24s} ROC_AUC={roc:.3f} PR_AUC={ap:.3f}")

if roc_list:
    print(f"overall ROC_AUC={float(np.mean(roc_list)):.3f} overall PR_AUC={float(np.mean(ap_list)):.3f}")
else:
    print('overall ROC_AUC=NA overall PR_AUC=NA (no definable labels)')

print('--- Binary (tuned thresholds from val; optimize F1) ---')
if 'best_thresholds' not in globals() or not isinstance(best_thresholds, dict) or len(best_thresholds) == 0:
    print('No tuned thresholds found. Run the threshold tuning cell above.')
else:
    f1_list = []
    for i, name in enumerate(hard_bin_names):
        thr = best_thresholds.get(name, None)
        if thr is None or not np.isfinite(thr):
            print(f"{name:24s} thr=NA (not tuned / single-class on val)")
            continue

        y_prob = pred_bin_aligned[:, i]
        y_hat = (y_prob >= thr).astype(int)
        y_true = y_bin_true[:, i]
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_hat, average='binary', zero_division=0)
        f1_list.append(float(f1))
        print(f"{name:24s} thr={thr:.3f} P={p:.2f} R={r:.2f} F1={f1:.2f}")

    if f1_list:
        print(f"overall F1={float(np.mean(f1_list)):.3f} (over tuned/defined labels)")
    else:
        print('overall F1=NA (no definable labels)')

print('--- Shade (auxiliary diagnostics) ---')
shade_pred = pred_shade.argmax(axis=1)
shade_acc = accuracy_score(y_shade_true, shade_pred)
print(f"Shade accuracy (overall): {shade_acc:.3f}")

shade_acc_conditional = float('nan')
shade_acc_paths_present = float('nan')

if 'walking_paths_p' in test_df.columns:
    w = test_df['walking_paths_p'].fillna(0.0).to_numpy(dtype=np.float32)
elif 'walking_paths' in test_df.columns:
    w = test_df['walking_paths'].fillna(0).to_numpy(dtype=np.float32)
else:
    w = None

if w is not None:
    shade_correct = (shade_pred == y_shade_true).astype(np.float32)
    denom = float(np.sum(w))
    if denom > 1e-8:
        shade_acc_conditional = float(np.sum(w * shade_correct) / denom)

    mask = w >= 0.5
    if int(np.sum(mask)) > 0:
        shade_acc_paths_present = float(accuracy_score(y_shade_true[mask], shade_pred[mask]))

print(
    f"Shade accuracy (conditional; weighted by walking_paths_p): {shade_acc_conditional:.3f}"
    if np.isfinite(shade_acc_conditional) else
    'Shade accuracy (conditional; weighted by walking_paths_p): NA'
)
print(
    f"Shade accuracy (paths present; walking_paths_p>=0.5): {shade_acc_paths_present:.3f}"
    if np.isfinite(shade_acc_paths_present) else
    'Shade accuracy (paths present; walking_paths_p>=0.5): NA'
)


Computed split-wise evaluation tables.
Threshold coverage (val-tuned labels): 8 / 8

--- Overall metrics by split ---


,split,n_samples,n_labels,overall_F1@0.5,overall_ROC_AUC,overall_PR_AUC,overall_F1@tuned,shade_acc_overall,shade_acc_conditional,shade_acc_paths_present
0,train,2347,8,0.7202,0.9465,0.8226,0.7603,0.9105,0.9545,0.9549
1,val,782,8,0.6078,0.8874,0.7065,0.6991,0.6662,0.6320,0.6345
2,test,782,8,0.6013,0.8789,0.6948,0.6713,0.6867,0.6723,0.6735



--- Per-label metrics by split ---


,split,label,support_pos,support_neg,P@0.5,R@0.5,F1@0.5,ROC_AUC,PR_AUC,thr_val,P@thr,R@thr,F1@thr
0,train,multipurpose_open_area,1853,494,0.9711,0.9239,0.9469,0.9710,0.9917,0.0877,0.9143,0.9849,0.9483
1,train,walking_paths,1750,597,0.9570,0.9023,0.9288,0.9603,0.9860,0.1543,0.9033,0.9714,0.9361
2,train,sports_field,635,1712,0.8880,0.8992,0.8936,0.9810,0.9607,0.6031,0.9205,0.8567,0.8874
3,train,built_structures,962,1385,0.8916,0.8212,0.8550,0.9567,0.9381,0.2726,0.8379,0.8971,0.8665
4,train,parking_lots,701,1646,0.8685,0.7347,0.7960,0.9542,0.9039,0.2480,0.7789,0.8745,0.8239
5,train,water_feature,478,1869,0.8765,0.5941,0.7082,0.9283,0.8317,0.3107,0.7923,0.6862,0.7354
6,train,children_s_playground,280,2067,0.6818,0.3750,0.4839,0.9181,0.6236,0.1464,0.4212,0.8107,0.5543
7,train,gardens,83,2264,0.6364,0.0843,0.1489,0.9024,0.3452,0.0905,0.2241,0.6265,0.3302
8,val,multipurpose_open_area,624,158,0.9428,0.8718,0.9059,0.9100,0.9668,0.0877,0.9118,0.9776,0.9435
9,val,walking_paths,580,202,0.9185,0.8552,0.8857,0.8956,0.9476,0.1543,0.8855,0.9603,0.9214


--- Binary (threshold=0.5) ---
sports_field             P=0.78 R=0.78 F1=0.78
multipurpose_open_area   P=0.93 R=0.89 F1=0.91
children_s_playground    P=0.55 R=0.20 F1=0.30
water_feature            P=0.60 R=0.39 F1=0.47
gardens                  P=0.00 R=0.00 F1=0.00
walking_paths            P=0.93 R=0.86 F1=0.90
built_structures         P=0.79 R=0.72 F1=0.76
parking_lots             P=0.78 R=0.63 F1=0.70
overall F1 (threshold=0.5) = 0.601
--- Binary (AUC) ---
sports_field             ROC_AUC=0.937 PR_AUC=0.863
multipurpose_open_area   ROC_AUC=0.910 PR_AUC=0.963
children_s_playground    ROC_AUC=0.882 PR_AUC=0.474
water_feature            ROC_AUC=0.787 PR_AUC=0.575
gardens                  ROC_AUC=0.785 PR_AUC=0.105
walking_paths            ROC_AUC=0.910 PR_AUC=0.965
built_structures         ROC_AUC=0.905 PR_AUC=0.815
parking_lots             ROC_AUC=0.916 PR_AUC=0.798
overall ROC_AUC=0.879 overall PR_AUC=0.695
--- Binary (tuned thresholds from val; optimize F1) ---
sports_field          

In [8]:
# Save split-wise evaluation tables (overall + per-label)
from pathlib import Path
from datetime import datetime

out_dir = Path('../report_outputs').resolve()
out_dir.mkdir(parents=True, exist_ok=True)

tag = globals().get('run_tag', None) or datetime.now().strftime('%Y%m%d_%H%M%S')

assert 'overall_split_df' in globals(), 'Run split-wise evaluation cell first (overall_split_df missing).'
assert 'per_label_split_df' in globals(), 'Run split-wise evaluation cell first (per_label_split_df missing).'

overall_path = out_dir / f"eval_overall_binary_ablation_{tag}.csv"
per_label_path = out_dir / f"eval_per_label_binary_ablation_{tag}.csv"

overall_split_df.to_csv(overall_path, index=False)
per_label_split_df.to_csv(per_label_path, index=False)

print('Saved overall table to', overall_path)
print('Saved per-label table to', per_label_path)

Saved overall table to /Users/starsrain/2025_codeProject/GreenSpace_CNN/report_outputs/eval_overall_binary_ablation_binary_ablation_20260223_201201.csv
Saved per-label table to /Users/starsrain/2025_codeProject/GreenSpace_CNN/report_outputs/eval_per_label_binary_ablation_binary_ablation_20260223_201201.csv
